In [1]:
from dotenv import load_dotenv

load_dotenv()

True

## Minimal multi-agent wiring

This is the smallest useful pattern: two trivial subagents and one supervisor. The goal is to see the wiring, not the complex work.

In [3]:
from langchain.tools import tool

@tool
def add_one(x: float) -> float:
    """Add one to a number."""
    return x + 1

@tool
def double(x: float) -> float:
    """Double a number."""
    return x * 2

In [4]:
from langchain.agents import create_agent

subagent_a = create_agent(
    model="groq:openai/gpt-oss-20b",
    tools=[add_one],
    system_prompt="You are a tiny helper. Always do one simple arithmetic step.",
)

subagent_b = create_agent(
    model="groq:openai/gpt-oss-20b",
    tools=[double],
    system_prompt="You are a tiny helper. Always do one simple arithmetic step.",
)

## Supervisor calls the subagents

In [5]:
from langchain.messages import HumanMessage

@tool
def call_subagent_a(x: float) -> float:
    """Ask the first small helper to add one."""
    response = subagent_a.invoke({
        "messages": [HumanMessage(content=f"Add one to {x}.")]
    })
    return float(response["messages"][-1].content)

@tool
def call_subagent_b(x: float) -> float:
    """Ask the second small helper to double the value."""
    response = subagent_b.invoke({
        "messages": [HumanMessage(content=f"Double {x}.")]
    })
    return float(response["messages"][-1].content)

supervisor = create_agent(
    model="groq:openai/gpt-oss-20b",
    tools=[call_subagent_a, call_subagent_b],
    system_prompt="You are a supervisor. Pick the right small helper and respond with the result.",
)

## Print the messages, then inspect a trace

The raw printout is useful, but it hides the internal tool-call chain. The trace shows exactly how the supervisor handed work to a subagent and how the state moved between calls.

In [6]:
question = "Add one to 5, then double it."

response = supervisor.invoke({"messages": [HumanMessage(content=question)]})

print("RAW RESPONSE")
print(response["messages"][-1].content)
print("\nMESSAGE LIST")
for msg in response["messages"]:
    print(type(msg).__name__, "-", getattr(msg, "content", ""))

RAW RESPONSE
12

MESSAGE LIST
HumanMessage - Add one to 5, then double it.
AIMessage - 
ToolMessage - 6.0
AIMessage - 
ToolMessage - 12.0
AIMessage - 12


In [7]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='Add one to 5, then double it.', additional_kwargs={}, response_metadata={}, id='ac8a8e04-badf-48ed-b79a-91d8a4da23cb'),
              AIMessage(content='', additional_kwargs={'reasoning_content': "We need to use the small helpers: first add one to 5, then double. The first helper is call_subagent_a, which adds one. The second helper is call_subagent_b, which doubles. We need to pick the right small helper and respond with the result. So we need to call call_subagent_a with x=5, get result 6. Then call call_subagent_b with x=6, get 12. Then respond with 12. We need to actually call the functions via the tool. According to instructions, we must pick the right small helper and respond with the result. So we will call call_subagent_a first, then call_subagent_b, then output the result. Let's do that.", 'tool_calls': [{'id': 'fc_4dfc12cd-71ce-4975-a384-507118e6475d', 'function': {'arguments': '{"x":5}', 'name': 'call_subagent_a'}, 'type': 'function'}]}, r

In [8]:
# A real split: the original personal chef both researched recipes and planned the response.
# Move recipe research into a dedicated subagent, leaving the planner responsible for the user-facing answer.

from typing import Any
from tavily import TavilyClient

chef_search_client = TavilyClient()

@tool
def search_recipes(query: str) -> dict[str, Any]:
    """Search for recipes matching the user's leftover ingredients."""
    return chef_search_client.search(query)

recipe_research_agent = create_agent(
    model="groq:openai/gpt-oss-20b",
    tools=[search_recipes],
    system_prompt="You are the recipe researcher. Search for recipes that use the ingredients provided and return concise source-backed findings. Do not write the final meal plan.",
)

@tool
def call_recipe_research(ingredients: str) -> str:
    """Delegate recipe research to the research subagent."""
    result = recipe_research_agent.invoke({
        "messages": [HumanMessage(content=f"Find recipe options using these leftovers: {ingredients}")]
    })
    return result["messages"][-1].content

meal_planner_agent = create_agent(
    model="groq:openai/gpt-oss-20b",
    tools=[call_recipe_research],
    system_prompt="You are a personal chef. Delegate recipe research, then turn the findings into three concise meal suggestions. Do not search the web yourself.",
)

meal_response = meal_planner_agent.invoke({
    "messages": [HumanMessage(content="I have leftover chicken and rice. Suggest three meals.")]
})

print(meal_response["messages"][-1].content)

Here are **three quick, tasty ways to turn your leftover chicken and rice into fresh meals** (based on the research I just ran):

| # | Meal | What it’s about | Key flavor notes |
|---|------|-----------------|------------------|
| 1 | **Chicken‑Fried Rice** | Stir‑fry the rice and chicken with peas, carrots, green onions, garlic, soy sauce, and a splash of sesame oil. Finish with a quick egg scramble if you like. | Savory, umami‑rich with a hint of nutty sesame. |
| 2 | **One‑Pot Chicken & Rice** | Warm the chicken and rice in a skillet with butter, salt, pepper, and optional herbs. It’s a single‑pan, no‑mess option. | Comfortingly simple, buttery, with a touch of fresh herbs. |
| 3 | **Creamy Chicken & Rice Casserole** | Mix the chicken and rice with a cream‑based sauce (e.g., cream of celery or chicken), mayo, and top with a breadcrumb‑butter crust. Bake until golden and bubbly. | Rich, creamy, with a crunchy, buttery topping. |

Feel free to pick the one that matches your time and 